In [ ]:
#Librerías y/o paquetrías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt #Visualizacion de datos
import yfinance as yf #Api de Yahoo Finanzas
#! pip install streamlit -q #Visualización en streamlit
from scipy.stats import kurtosis, skew, shapiro ,norm,t #Funciones estadísticas
#import Funciones_MCF as MCF #Utilizar solo si se estás trabajando en VSC y tienes las funciones en un archivo a parte
from datetime import date

'''
Para usar streamlit en google colab es un poco más de desmadre
pq no usas la terminal directamente como en VSC
tons no lo haré ahorita, pero de acá sale
https://medium.com/@yash.kavaiya3/running-streamlit-code-in-google-colab-involves-a-few-steps-c43ea0e8c0d9#:~:text=La%20segunda%20l%C3%ADnea%20(%20!,externa%20usando%20el%20comando%20wget%20.&text=La%20l%C3%ADnea%20%25%25writefile%20app,contactarnos%20si%20tienes%20alguna%20duda.
Chance pa eso sí conviene más el VSC, como q siento se entiende mejor
'''

'\nPara usar streamlit en google colab es un poco más de desmadre\npq no usas la terminal directamente como en VSC\ntons no lo haré ahorita, pero de acá sale\nhttps://medium.com/@yash.kavaiya3/running-streamlit-code-in-google-colab-involves-a-few-steps-c43ea0e8c0d9#:~:text=La%20segunda%20l%C3%ADnea%20(%20!,externa%20usando%20el%20comando%20wget%20.&text=La%20l%C3%ADnea%20%25%25writefile%20app,contactarnos%20si%20tienes%20alguna%20duda.\nChance pa eso sí conviene más el VSC, como q siento se entiende mejor\n'

In [ ]:
#Funciones a ultilizar
def obtener_datos(stocks):
    '''
    El objetivo de esta funcion es descargar el precio
    de cierre de uno o varios activos en una ventana de 2010 a dia de hoy

    Input = Ticker del activo en string
    Output = DataFrame del precio del activo

    '''
    df = yf.download(stocks, start = "2010-01-01" , end = date.today())['Close']
    return df


def calcular_rendimientos(df): #Función de rendimientos simples
    '''
    Funcion de calcula los rendimientos de un activo

    Input = Data Frame de precios por activo

    Output = Data Frame de  rendimientos

    '''
    return df.pct_change().dropna()


def calcular_rendimientos_Log(df): #Función de rendimientos logarítmicos
    '''
    Funcion que calcula los rendimientos de un activo

    Input = Data Frame de precios por activo

    Output = Data Frame de  rendimientos

    '''
    rendimientos = np.log(df.iloc[:,0]) - np.log(df.iloc[:,0].shift(1))

    return rendimientos.dropna()

#*Código principal*#

In [ ]:
#Código
#inciso a) Descarga de información de un activo
ticker = "NVDA"
df_precios = obtener_datos(ticker)
df_precios.iloc[:,0]
empresa = yf.Ticker(ticker)
nombre = empresa.info['longName']

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed


In [ ]:
#b) Calcular rendimientos
df_rendimientos_log = calcular_rendimientos_Log(df_precios)
#df_rendimientos_log = calcular_rendimientos(df_precios)
print(df_rendimientos_log)

#media
media = df_rendimientos_log.mean()
print(media)

#sesgo
sesgo = skew(df_rendimientos_log)
print(sesgo)

#curtosis
curtosis = kurtosis(df_rendimientos_log)
print(curtosis)

Date
2010-01-05    0.014497
2010-01-06    0.006376
2010-01-07   -0.019792
2010-01-08    0.002159
2010-01-11   -0.014115
                ...   
2025-03-24    0.031034
2025-03-25   -0.005948
2025-03-26   -0.059134
2025-03-27   -0.020694
2025-03-28   -0.015921
Name: NVDA, Length: 3832, dtype: float64
0.0014498360930741304
0.14491247784035913
6.796189095963291


In [ ]:

#VaR método paramétrico Normal--------------------------------------------------------------------------------------------------------------------------
mu = media
sigma = np.std(df_rendimientos_log)

VaR_95 = norm.ppf(1-0.95,mu,sigma)

print(type(VaR_95))
print(type(mu))
print(type(sigma))

<class 'numpy.float64'>
<class 'numpy.float64'>
<class 'float'>


In [ ]:
'''
c)
Calcula el VaR y ES para la serie completa de datos a los siguientes intervalos de confianza:
α = 0,95, 0,975, y 0,99 bajo una aproximación param ́etrica asumiendo una distribuci ́on normal y t-student,
además bajo una aproximación histórica y Monte Carlo.
'''
#VaR método paramétrico Normal--------------------------------------------------------------------------------------------------------------------------
mu = media
sigma = np.std(df_rendimientos_log)

VaR_95 = norm.ppf(1-0.95,mu,sigma)
VaR_975 = norm.ppf(1-0.975,mu,sigma)
VaR_99 = norm.ppf(1-0.99,mu,sigma)

print("El 95% VaR de", nombre , "es:", round(VaR_95*100,4))
print("El 975% VaR de", nombre , "es:", round(VaR_975*100,4))
print("El 99% VaR de", nombre , "es:", round(VaR_99*100,4))

#VaR método paramétrico T-student
gl = len(df_rendimientos_log)-1
VaR_95_t = t.ppf(1-0.95,gl,mu,sigma)
VaR_975_t = t.ppf(1-0.975,gl,mu,sigma)
VaR_99_t = t.ppf(1-0.99,gl,mu,sigma)

print("\nEl 95% VaR de", nombre , "es:", round(VaR_95_t*100,4))
print("El 975% VaR de", nombre , "es:", round(VaR_975_t*100,4))
print("El 99% VaR de", nombre , "es:", round(VaR_99_t*100,4))

#VaR histórico
hVaR_95 = df_rendimientos_log.quantile(0.05)
hVaR_975 = df_rendimientos_log.quantile(0.025)
hVaR_99 = df_rendimientos_log.quantile(0.001)

print("\nEl 95% VaR de", nombre , "es:", round(hVaR_95*100,4))
print("El 975% VaR de", nombre , "es:", round(hVaR_975*100,4))
print("El 99% VaR de", nombre , "es:", round(hVaR_99*100,4))

#VaR Monte Carlo         Está en normal pero chance podemos cambiarla a una t
n_sim = 1000000
sim_returns = np.random.normal(mu,sigma ,n_sim)

MCVaR_95 = np.percentile(sim_returns, 5)
MCVaR_975 = np.percentile(sim_returns, 2.5)
MCVaR_99 = np.percentile(sim_returns, 0.1)

print("\nEl 95% VaR de", nombre , "es:", round(MCVaR_95*100,4))
print("El 975% VaR de", nombre , "es:", round(MCVaR_975*100,4))
print("El 99% VaR de", nombre , "es:", round(MCVaR_99*100,4))

El 95% VaR de NVIDIA Corporation es: -4.5775
El 975% VaR de NVIDIA Corporation es: -5.4823
El 99% VaR de NVIDIA Corporation es: -6.5342

El 95% VaR de NVIDIA Corporation es: -4.5787
El 975% VaR de NVIDIA Corporation es: -5.484
El 99% VaR de NVIDIA Corporation es: -6.537

El 95% VaR de NVIDIA Corporation es: -4.3298
El 975% VaR de NVIDIA Corporation es: -5.7703
El 99% VaR de NVIDIA Corporation es: -13.3613

El 95% VaR de NVIDIA Corporation es: -4.5861
El 975% VaR de NVIDIA Corporation es: -5.4944
El 99% VaR de NVIDIA Corporation es: -8.7418


In [ ]:
#ES paramétrico normal
ES_95 = df_rendimientos_log[df_rendimientos_log<= VaR_95].mean()
ES_975 = df_rendimientos_log[df_rendimientos_log<= VaR_975].mean()
ES_99 = df_rendimientos_log[df_rendimientos_log<= VaR_99].mean()

print("\nEl 95% Expected Shorfall de", nombre , "es:", round(ES_95*100,4))
print("El 975% Expected Shorfall de", nombre , "es:", round(ES_975*100,4))
print("El 99% Expected Shorfall de", nombre , "es:", round(ES_99*100,4))

#ES histórico
hES_95 = df_rendimientos_log[df_rendimientos_log<= hVaR_95].mean()
hES_975 = df_rendimientos_log[df_rendimientos_log<= hVaR_975].mean()
hES_99 = df_rendimientos_log[df_rendimientos_log<= hVaR_99].mean()

print("\nEl 95% Expected Shorfall de", nombre , "es:", round(hES_95*100,4))
print("El 975% Expected Shorfall de", nombre , "es:", round(hES_975*100,4))
print("El 99% Expected Shorfall de", nombre , "es:", round(hES_99*100,4))

#ES parámetrico T-student
ES_95_t = df_rendimientos_log[df_rendimientos_log<= VaR_95_t].mean()
ES_975_t = df_rendimientos_log[df_rendimientos_log<= VaR_975_t].mean()
ES_99_t = df_rendimientos_log[df_rendimientos_log<= VaR_99_t].mean()

print("\nEl 95% Expected Shorfall de", nombre , "es:", round(ES_95_t*100,4))
print("El 975% Expected Shorfall de", nombre , "es:", round(ES_975_t*100,4))
print("El 99% Expected Shorfall de", nombre , "es:", round(ES_99_t*100,4))

#ES Monte Carlo
MCES_95 = df_rendimientos_log[df_rendimientos_log<= MCVaR_95].mean()
MCES_975 = df_rendimientos_log[df_rendimientos_log<= MCVaR_975].mean()
MCES_99 = df_rendimientos_log[df_rendimientos_log<= MCVaR_99].mean()

print("\nEl 95% Expected Shorfall de", nombre , "es:", round(MCES_95*100,4))
print("El 975% Expected Shorfall de", nombre , "es:", round(MCES_975*100,4))
print("El 99% Expected Shorfall de", nombre , "es:", round(MCES_99*100,4))


El 95% Expected Shorfall de NVIDIA Corporation es: -6.7412
El 975% Expected Shorfall de NVIDIA Corporation es: -7.7336
El 99% Expected Shorfall de NVIDIA Corporation es: -8.8305

El 95% Expected Shorfall de NVIDIA Corporation es: -6.4558
El 975% Expected Shorfall de NVIDIA Corporation es: -7.9876
El 99% Expected Shorfall de NVIDIA Corporation es: -18.6605

El 95% Expected Shorfall de NVIDIA Corporation es: -6.7412
El 975% Expected Shorfall de NVIDIA Corporation es: -7.7336
El 99% Expected Shorfall de NVIDIA Corporation es: -8.8305

El 95% Expected Shorfall de NVIDIA Corporation es: -6.7412
El 975% Expected Shorfall de NVIDIA Corporation es: -7.7546
El 99% Expected Shorfall de NVIDIA Corporation es: -11.868


##Inciso d)##
En una sola gráfica muestra las ganancias y ṕerdidas además del VaR y el ES con α = 0,95 y 0,99 con una rolling window de 252 retornos.

In [ ]:
# VaR Rolling método paramétrico Normal
mu_roll = df_rendimientos_log.rolling(window=252).mean()
sigma_roll = df_rendimientos_log.rolling(window=252).std()

VaR_95_rolling = norm.ppf(1-0.95, mu_roll, sigma_roll)
VaR_99_rolling = norm.ppf(1-0.99, mu_roll, sigma_roll)

VaR_95_rolling_porcentaje = (VaR_95_rolling * 100).round(4)
VaR_99_rolling_porcentaje = (VaR_99_rolling * 100).round(4)

VaR_95_rolling_df = pd.DataFrame({'Fecha': df_rendimientos_log.index, '95% VaR Rolling': VaR_95_rolling_porcentaje.squeeze()})
VaR_99_rolling_df = pd.DataFrame({'Fecha': df_rendimientos_log.index, '99% VaR Rolling': VaR_99_rolling_porcentaje.squeeze()})

print(VaR_95_rolling_df.tail())
### print(vaR_99_rolling_df.tail()) ###

#VaR Rolling histórico
hVaR_95_r = df_rendimientos_log.rolling(window=252).quantile(0.05)
hVaR_99_r = df_rendimientos_log.rolling(window=252).quantile(0.001)

hVaR_95_r_porcentaje = (hVaR_95_r * 100).round(4)
hVaR_99_r_porcentaje = (hVaR_99_r * 100).round(4)

hVaR_95_r_df = pd.DataFrame({'Fecha': df_rendimientos_log.index, '95% hVaR Rolling': hVaR_95_r_porcentaje.squeeze()})
hVaR_99_r_df = pd.DataFrame({'Fecha': df_rendimientos_log.index, '99% hVaR Rolling': hVaR_99_r_porcentaje.squeeze()})

print(hVaR_95_r_df.tail())
### print(hVaR_99_r_df.tail()) ###

          Fecha  95% VaR Rolling
3827 2025-03-24          -5.7482
3828 2025-03-25          -5.7547
3829 2025-03-26          -5.8153
3830 2025-03-27          -5.8302
3831 2025-03-28          -5.8222
                Fecha  95% hVaR Rolling
Date                                   
2025-03-24 2025-03-24           -6.1381
2025-03-25 2025-03-25           -6.1381
2025-03-26 2025-03-26           -6.1408
2025-03-27 2025-03-27           -6.1408
2025-03-28 2025-03-28           -6.1408


In [ ]:
#ES Rolling paramétrico normal
ES_95_rolling = df_rendimientos_log[df_rendimientos_log<= VaR_95_rolling].mean()
ES_99_rolling = df_rendimientos_log[df_rendimientos_log<= VaR_99_rolling].mean()

ES_95_roll_porcentaje = (ES_95_rolling * 100).round(4)
ES_99_roll_porcentaje = (ES_99_rolling * 100).round(4)

ES_95_roll_df = pd.DataFrame({'Fecha': df_rendimientos_log.index, '95% ES Rolling': ES_95_roll_porcentaje.squeeze()})
ES_99_roll_df = pd.DataFrame({'Fecha': df_rendimientos_log.index, '99% ES Rolling': ES_99_roll_porcentaje.squeeze()})

print(ES_95_roll_df.tail())
### print(ES_99_roll_df.tail()) ###

#ES Rolling histórico
hES_95_rolling = df_rendimientos_log[df_rendimientos_log<= hVaR_95_r].mean()
hES_99_rolling = df_rendimientos_log[df_rendimientos_log<= hVaR_99_r].mean()

hES_95_r_porcentaje = (hES_95_rolling * 100).round(4)
hES_99_r_porcentaje = (hES_99_rolling * 100).round(4)

hES_95_r_df = pd.DataFrame({'Fecha': df_rendimientos_log.index, '95% hES Rolling': hES_95_r_porcentaje.squeeze()})
hES_99_r_df = pd.DataFrame({'Fecha': df_rendimientos_log.index, '99% hES Rolling': hES_99_r_porcentaje.squeeze()})

print(hES_95_r_df.tail())
### print(hVaR_99_r_df.tail()) ###

          Fecha  95% ES Rolling
3827 2025-03-24         -6.1973
3828 2025-03-25         -6.1973
3829 2025-03-26         -6.1973
3830 2025-03-27         -6.1973
3831 2025-03-28         -6.1973
          Fecha  95% hES Rolling
3827 2025-03-24          -5.8162
3828 2025-03-25          -5.8162
3829 2025-03-26          -5.8162
3830 2025-03-27          -5.8162
3831 2025-03-28          -5.8162


#*Streamlit*#

In [ ]:
#Streamlit